
# 03M — Corrected Unit Evidence Rule Audit

**Notebook build ID: `03M_UNIT_EVIDENCE_RULE_AUDIT_V1`**

03L established the corrected spatial representation:

\[
\boxed{
L_{\max}=250\text{ m},
\qquad
g_{\max}=10\text{ m},
\qquad
\text{within-unit sampling}\approx25\text{ m}
}
\]

and successfully reconstructed cached NIROPS evidence for:

- 415 incidents,
- 572,200 corrected local units,
- about 4.07 million within-unit sample points.

03M performs the **last target-definition audit** before the corrected ML cohort is frozen.

---

# Why 03M is needed

The 03L primary rule is intentionally simple:

### Burned Over

\[
\text{supported if any sampled location shows a supported crossing}.
\]

### Held

\[
\text{supported if at least one sampled location has supported approach}
\]

and

\[
\text{no sampled location is a Held trajectory conflict}.
\]

The rule has correct event semantics, but the probability of triggering an `any` or `zero-conflict` condition can change with the number of sample points.

A longer unit with more sampled locations has more opportunities to observe:

- at least one crossing;
- at least one conflict.

03M therefore separates **physical line coverage** from **raw point count**.

---

# Arc-length weighting

For geometry part \(j\) with length \(L_j\) represented by \(n_j\) sample points, every sample receives weight:

\[
w_{jp}
=
\frac{L_j}{n_j}.
\]

Therefore:

\[
\sum_p w_p
=
L_{\mathrm{unit}}.
\]

Weighted support, conflict, and insufficient fractions are:

\[
f_S
=
\frac{
\sum_p
w_p
\mathbf{1}[p\text{ supported}]
}{
L_{\mathrm{unit}}
},
\]

\[
f_C
=
\frac{
\sum_p
w_p
\mathbf{1}[p\text{ conflict}]
}{
L_{\mathrm{unit}}
},
\]

and

\[
f_I = 1-f_S-f_C.
\]

A 2 m GIS fragment therefore contributes approximately 2 m of evidence rather than the same statistical vote as a 25 m line portion.

---

# Main audits

## 1. Opportunity-bias diagnostic

For Held and Burned Over separately, report the 03L primary support rate by:

### Sample count

- 1
- 2
- 3–4
- 5–7
- 8–10
- >10

### Unit length

- <25 m
- 25–50 m
- 50–100 m
- 100–150 m
- 150–200 m
- 200–250 m

The tables also report weighted support/conflict fractions.

These are diagnostics, not causal tests.

---

## 2. Weighted evidence sensitivity

Candidate support thresholds:

\[
f_S\ge
\{0.05,0.10,0.20,0.25,0.50\}.
\]

For Held, candidate conflict tolerances are:

\[
f_C\le
\{0,0.05,0.10,0.20\}.
\]

For Burned Over, the main requirement is sufficient weighted crossing coverage; conflict tolerance is not used because a supported crossing directly corroborates a Burned Over outcome.

No threshold is automatically selected.

---

## 3. Challenge-interval dispersion

Different sample locations within one corrected unit can become supported in different NIROPS intervals.

03M therefore records:

- number of distinct supported intervals;
- earliest supported interval;
- dominant supported interval by arc-length weight;
- dominant interval weight fraction;
- total time span across supported intervals.

This determines whether HRRR should later use:

- one dominant interval,
- or a broader interval summary.

---

# Speed architecture

03M performs **no rasterization**.

It reuses only:

```text
units\
point_evidence\
unit_evidence\
trajectories\
```

from 03L.

Processing is incident-wise and resumable.

The expensive 4.07M-point table is never held in memory all at once.


In [ ]:
import os  # Configurable project root for the public release.

NOTEBOOK_BUILD_ID = "03M_UNIT_EVIDENCE_RULE_AUDIT_V1"

from pathlib import Path

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fireline_corrected_local_250m_10m"
)

UNIT_ROOT = (
    PROCESSED_ROOT
    / "units"
)

POINT_EVIDENCE_ROOT = (
    PROCESSED_ROOT
    / "point_evidence"
)

UNIT_EVIDENCE_ROOT = (
    PROCESSED_ROOT
    / "unit_evidence"
)

TRAJECTORY_ROOT = (
    PROCESSED_ROOT
    / "trajectories"
)

INCIDENT_META_ROOT = (
    PROCESSED_ROOT
    / "incident_metadata"
)

AUDIT_PROCESSED_ROOT = (
    PROCESSED_ROOT
    / "unit_evidence_rule_audit"
)

UNIT_METRIC_SHARD_ROOT = (
    AUDIT_PROCESSED_ROOT
    / "unit_metrics"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "unit_evidence_rule_audit"
)

for path in [
    AUDIT_PROCESSED_ROOT,
    UNIT_METRIC_SHARD_ROOT,
    OUTPUT_ROOT,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

assert UNIT_ROOT.exists(), UNIT_ROOT
assert POINT_EVIDENCE_ROOT.exists(), POINT_EVIDENCE_ROOT
assert UNIT_EVIDENCE_ROOT.exists(), UNIT_EVIDENCE_ROOT
assert TRAJECTORY_ROOT.exists(), TRAJECTORY_ROOT

SUPPORT_THRESHOLDS = [
    0.05,
    0.10,
    0.20,
    0.25,
    0.50,
]

HELD_CONFLICT_TOLERANCES = [
    0.00,
    0.05,
    0.10,
    0.20,
]

RESUME = True

print("=" * 84)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 84)


## 1. Imports

In [ ]:

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "pyarrow": "pyarrow",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

for module, package in required.items():

    if importlib.util.find_spec(
        module
    ) is None:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

import json
import gc
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.stats import spearmanr
from tqdm.auto import tqdm

pd.set_option(
    "display.max_columns",
    300,
)

pd.set_option(
    "display.max_rows",
    300,
)

pd.set_option(
    "display.width",
    220,
)

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)


## 2. Incident inventory and completeness

In [ ]:

unit_files = sorted(
    UNIT_ROOT.glob(
        "*.parquet"
    )
)

assert unit_files

inventory_rows = []

for unit_path in tqdm(
    unit_files,
    desc="03M incident inventory",
):

    irwin = (
        unit_path.stem
    )

    point_path = (
        POINT_EVIDENCE_ROOT
        / f"{irwin}.parquet"
    )

    evidence_path = (
        UNIT_EVIDENCE_ROOT
        / f"{irwin}.parquet"
    )

    trajectory_path = (
        TRAJECTORY_ROOT
        / f"{irwin}.npz"
    )

    if not (
        point_path.exists()
        and evidence_path.exists()
        and trajectory_path.exists()
    ):

        continue

    unit_preview = pd.read_parquet(
        unit_path,
        columns=[
            "_irwin_norm",
            "IncidentName",
            "Year",
            "unit_uid",
        ],
    )

    inventory_rows.append(
        {
            "_irwin_norm": str(
                unit_preview[
                    "_irwin_norm"
                ].iloc[
                    0
                ]
            ),
            "IncidentName": (
                unit_preview[
                    "IncidentName"
                ].dropna().iloc[
                    0
                ]
                if unit_preview[
                    "IncidentName"
                ].notna().any()
                else irwin
            ),
            "Year": (
                unit_preview[
                    "Year"
                ].dropna().iloc[
                    0
                ]
                if unit_preview[
                    "Year"
                ].notna().any()
                else pd.NA
            ),
            "units": len(
                unit_preview
            ),
            "unit_path": str(
                unit_path
            ),
            "point_path": str(
                point_path
            ),
            "evidence_path": str(
                evidence_path
            ),
            "trajectory_path": str(
                trajectory_path
            ),
        }
    )

incident_inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        "units"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Auditable incidents:",
    len(
        incident_inventory
    )
)

print(
    "Corrected units represented:",
    f"{incident_inventory['units'].sum():,}"
)

display(
    incident_inventory.head(
        20
    )
)



# 3. Part-length reconstruction

03L stores `part_index` for every sample point.

03M reconstructs the actual length of each unit geometry part by exploding the saved corrected unit geometry.

For each:

\[
(\text{unit},\text{part})
\]

the sample weight is:

\[
w=
\frac{
L_{\mathrm{part}}
}{
N_{\mathrm{samples\ in\ part}}
}.
\]

A strict validation confirms that weighted sample lengths sum back to the unit's stored actual line length.


In [ ]:

def build_part_length_table(
    units,
):

    source = (
        units[
            [
                "unit_row_index",
                "geometry",
            ]
        ]
        .set_index(
            "unit_row_index"
        )
    )

    exploded = source.explode(
        index_parts=True
    )

    exploded = (
        exploded.reset_index()
    )

    if (
        "level_1"
        in exploded.columns
    ):

        exploded = exploded.rename(
            columns={
                "level_1": "part_index",
            }
        )

    elif (
        "part_index"
        not in exploded.columns
    ):

        candidates = [
            column
            for column in exploded.columns
            if column
            not in {
                "unit_row_index",
                "geometry",
            }
        ]

        if len(
            candidates
        ) != 1:

            raise RuntimeError(
                "Could not identify exploded geometry part index."
            )

        exploded = exploded.rename(
            columns={
                candidates[
                    0
                ]: "part_index",
            }
        )

    exploded[
        "part_index"
    ] = (
        exploded[
            "part_index"
        ]
        .astype(
            np.int16
        )
    )

    exploded[
        "part_length_m"
    ] = (
        exploded.geometry.length
        .astype(
            float
        )
    )

    return exploded[
        [
            "unit_row_index",
            "part_index",
            "part_length_m",
        ]
    ]


## 4. Build weighted unit metrics for one incident

In [ ]:

def audit_incident(
    unit_path,
    point_path,
    evidence_path,
    trajectory_path,
):

    units = gpd.read_parquet(
        unit_path
    )

    point_table = pd.read_parquet(
        point_path
    )

    primary = pd.read_parquet(
        evidence_path
    )

    with np.load(
        trajectory_path,
        allow_pickle=False,
    ) as trajectory:

        times_ns = (
            trajectory[
                "times_ns"
            ]
            .copy()
        )

    times = pd.to_datetime(
        times_ns,
        utc=True,
    )

    # ------------------------------------------------------------
    # Structural checks
    # ------------------------------------------------------------

    assert units[
        "unit_uid"
    ].is_unique

    assert primary[
        "unit_uid"
    ].is_unique

    assert (
        len(
            units
        )
        == len(
            primary
        )
    )

    assert set(
        units[
            "unit_uid"
        ]
    ) == set(
        primary[
            "unit_uid"
        ]
    )

    part_lengths = build_part_length_table(
        units
    )

    sample_count_by_part = (
        point_table.groupby(
            [
                "unit_row_index",
                "part_index",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size": "samples_in_part",
            }
        )
    )

    weighted_points = (
        point_table
        .merge(
            part_lengths,
            on=[
                "unit_row_index",
                "part_index",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            sample_count_by_part,
            on=[
                "unit_row_index",
                "part_index",
            ],
            how="left",
            validate="many_to_one",
        )
    )

    if (
        weighted_points[
            "part_length_m"
        ].isna().any()
    ):

        raise RuntimeError(
            "Sample points could not be mapped to geometry-part length."
        )

    weighted_points[
        "sample_length_weight_m"
    ] = (
        weighted_points[
            "part_length_m"
        ]
        / weighted_points[
            "samples_in_part"
        ]
    )

    weighted_points[
        "_support_weight_m"
    ] = np.where(
        weighted_points[
            "point_evidence"
        ]
        == "temporally_supported",
        weighted_points[
            "sample_length_weight_m"
        ],
        0.0,
    )

    weighted_points[
        "_conflict_weight_m"
    ] = np.where(
        weighted_points[
            "point_evidence"
        ]
        == "trajectory_conflict",
        weighted_points[
            "sample_length_weight_m"
        ],
        0.0,
    )

    weighted_points[
        "_insufficient_weight_m"
    ] = np.where(
        weighted_points[
            "point_evidence"
        ]
        == "insufficient_temporal_evidence",
        weighted_points[
            "sample_length_weight_m"
        ],
        0.0,
    )

    weighted_summary = (
        weighted_points.groupby(
            "unit_row_index",
            as_index=False,
        )
        .agg(
            weighted_line_length_m=(
                "sample_length_weight_m",
                "sum",
            ),
            weighted_support_length_m=(
                "_support_weight_m",
                "sum",
            ),
            weighted_conflict_length_m=(
                "_conflict_weight_m",
                "sum",
            ),
            weighted_insufficient_length_m=(
                "_insufficient_weight_m",
                "sum",
            ),
            sample_points_recounted=(
                "sample_row_index",
                "size",
            ),
            geometry_parts=(
                "part_index",
                "nunique",
            ),
        )
    )

    unit_base = (
        pd.DataFrame(
            units.drop(
                columns="geometry"
            )
        )
        .merge(
            primary[
                [
                    "unit_uid",
                    "unit_evidence",
                    "unit_diagnostic_reason",
                    "sample_points",
                    "supported_points",
                    "conflict_points",
                    "insufficient_points",
                    "supported_fraction",
                    "conflict_fraction",
                    "insufficient_fraction",
                    "any_supported",
                    "majority_supported",
                    "all_supported",
                    "any_conflict",
                    "challenge_interval_start_utc",
                    "challenge_interval_end_utc",
                    "challenge_interval_hours",
                ]
            ],
            on="unit_uid",
            how="left",
            validate="one_to_one",
            suffixes=(
                "",
                "_primary",
            ),
        )
        .merge(
            weighted_summary,
            on="unit_row_index",
            how="left",
            validate="one_to_one",
        )
    )

    unit_base[
        "weighted_support_fraction"
    ] = (
        unit_base[
            "weighted_support_length_m"
        ]
        / unit_base[
            "weighted_line_length_m"
        ]
    )

    unit_base[
        "weighted_conflict_fraction"
    ] = (
        unit_base[
            "weighted_conflict_length_m"
        ]
        / unit_base[
            "weighted_line_length_m"
        ]
    )

    unit_base[
        "weighted_insufficient_fraction"
    ] = (
        unit_base[
            "weighted_insufficient_length_m"
        ]
        / unit_base[
            "weighted_line_length_m"
        ]
    )

    unit_base[
        "primary_supported"
    ] = (
        unit_base[
            "unit_evidence"
        ]
        == "temporally_supported"
    )

    # ------------------------------------------------------------
    # Weight conservation check
    # ------------------------------------------------------------

    unit_base[
        "weight_length_abs_error_m"
    ] = (
        unit_base[
            "weighted_line_length_m"
        ]
        - unit_base[
            "actual_line_length_m"
        ]
    ).abs()

    max_weight_error = float(
        unit_base[
            "weight_length_abs_error_m"
        ].max()
    )

    if (
        max_weight_error
        > 1e-5
    ):

        raise RuntimeError(
            "Arc-length weights do not conserve corrected unit length. "
            f"Max error={max_weight_error:.8f} m"
        )

    if not (
        unit_base[
            "sample_points"
        ].astype(
            int
        )
        == unit_base[
            "sample_points_recounted"
        ].astype(
            int
        )
    ).all():

        raise RuntimeError(
            "Point-evidence sample count differs from unit-evidence sample count."
        )

    # ------------------------------------------------------------
    # Challenge-interval dispersion
    # ------------------------------------------------------------

    supported_events = weighted_points[
        (
            weighted_points[
                "point_evidence"
            ]
            == "temporally_supported"
        )
        & (
            weighted_points[
                "event_start_idx"
            ]
            >= 0
        )
        & (
            weighted_points[
                "event_end_idx"
            ]
            >= 0
        )
    ][
        [
            "unit_row_index",
            "event_start_idx",
            "event_end_idx",
            "sample_length_weight_m",
        ]
    ].copy()

    if len(
        supported_events
    ):

        interval_weights = (
            supported_events.groupby(
                [
                    "unit_row_index",
                    "event_start_idx",
                    "event_end_idx",
                ],
                as_index=False,
            )
            .agg(
                interval_supported_weight_m=(
                    "sample_length_weight_m",
                    "sum",
                ),
                interval_supported_points=(
                    "sample_length_weight_m",
                    "size",
                ),
            )
        )

        interval_stats = (
            interval_weights.groupby(
                "unit_row_index",
                as_index=False,
            )
            .agg(
                supported_interval_count=(
                    "event_end_idx",
                    "size",
                ),
                earliest_supported_start_idx=(
                    "event_start_idx",
                    "min",
                ),
                latest_supported_end_idx=(
                    "event_end_idx",
                    "max",
                ),
                total_supported_event_weight_m=(
                    "interval_supported_weight_m",
                    "sum",
                ),
            )
        )

        dominant = (
            interval_weights.sort_values(
                [
                    "unit_row_index",
                    "interval_supported_weight_m",
                    "event_end_idx",
                    "event_start_idx",
                ],
                ascending=[
                    True,
                    False,
                    True,
                    True,
                ],
            )
            .drop_duplicates(
                "unit_row_index",
                keep="first",
            )
            .rename(
                columns={
                    "event_start_idx": "dominant_supported_start_idx",
                    "event_end_idx": "dominant_supported_end_idx",
                    "interval_supported_weight_m": "dominant_supported_weight_m",
                    "interval_supported_points": "dominant_supported_points",
                }
            )
        )

        challenge = (
            interval_stats
            .merge(
                dominant[
                    [
                        "unit_row_index",
                        "dominant_supported_start_idx",
                        "dominant_supported_end_idx",
                        "dominant_supported_weight_m",
                        "dominant_supported_points",
                    ]
                ],
                on="unit_row_index",
                how="left",
                validate="one_to_one",
            )
        )

        challenge[
            "dominant_interval_weight_fraction"
        ] = (
            challenge[
                "dominant_supported_weight_m"
            ]
            / challenge[
                "total_supported_event_weight_m"
            ]
        )

        challenge[
            "earliest_supported_interval_start_utc"
        ] = [
            times[
                int(
                    value
                )
            ]
            for value in (
                challenge[
                    "earliest_supported_start_idx"
                ]
            )
        ]

        challenge[
            "latest_supported_interval_end_utc"
        ] = [
            times[
                int(
                    value
                )
            ]
            for value in (
                challenge[
                    "latest_supported_end_idx"
                ]
            )
        ]

        challenge[
            "dominant_supported_interval_start_utc"
        ] = [
            times[
                int(
                    value
                )
            ]
            for value in (
                challenge[
                    "dominant_supported_start_idx"
                ]
            )
        ]

        challenge[
            "dominant_supported_interval_end_utc"
        ] = [
            times[
                int(
                    value
                )
            ]
            for value in (
                challenge[
                    "dominant_supported_end_idx"
                ]
            )
        ]

        challenge[
            "supported_event_span_hours"
        ] = (
            pd.to_datetime(
                challenge[
                    "latest_supported_interval_end_utc"
                ],
                utc=True,
            )
            - pd.to_datetime(
                challenge[
                    "earliest_supported_interval_start_utc"
                ],
                utc=True,
            )
        ).dt.total_seconds() / 3600.0

        challenge_columns = [
            "unit_row_index",
            "supported_interval_count",
            "earliest_supported_interval_start_utc",
            "latest_supported_interval_end_utc",
            "dominant_supported_interval_start_utc",
            "dominant_supported_interval_end_utc",
            "dominant_supported_weight_m",
            "total_supported_event_weight_m",
            "dominant_interval_weight_fraction",
            "supported_event_span_hours",
        ]

        unit_base = (
            unit_base.merge(
                challenge[
                    challenge_columns
                ],
                on="unit_row_index",
                how="left",
                validate="one_to_one",
            )
        )

    else:

        unit_base[
            "supported_interval_count"
        ] = 0

        for column in [
            "earliest_supported_interval_start_utc",
            "latest_supported_interval_end_utc",
            "dominant_supported_interval_start_utc",
            "dominant_supported_interval_end_utc",
        ]:

            unit_base[
                column
            ] = pd.NaT

        for column in [
            "dominant_supported_weight_m",
            "total_supported_event_weight_m",
            "dominant_interval_weight_fraction",
            "supported_event_span_hours",
        ]:

            unit_base[
                column
            ] = np.nan

    unit_base[
        "supported_interval_count"
    ] = (
        unit_base[
            "supported_interval_count"
        ]
        .fillna(
            0
        )
        .astype(
            np.int16
        )
    )

    keep_columns = [
        "unit_uid",
        "unit_row_index",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "FeatureCategory",
        "_engagement_norm",
        "_geom_hash",
        "source_index",
        "unit_origin",
        "actual_line_length_m",
        "constituent_components",
        "max_gap_used_m",
        "sample_points",
        "supported_points",
        "conflict_points",
        "insufficient_points",
        "supported_fraction",
        "conflict_fraction",
        "insufficient_fraction",
        "primary_supported",
        "unit_evidence",
        "unit_diagnostic_reason",
        "weighted_support_length_m",
        "weighted_conflict_length_m",
        "weighted_insufficient_length_m",
        "weighted_support_fraction",
        "weighted_conflict_fraction",
        "weighted_insufficient_fraction",
        "geometry_parts",
        "weight_length_abs_error_m",
        "challenge_interval_start_utc",
        "challenge_interval_end_utc",
        "challenge_interval_hours",
        "supported_interval_count",
        "earliest_supported_interval_start_utc",
        "latest_supported_interval_end_utc",
        "dominant_supported_interval_start_utc",
        "dominant_supported_interval_end_utc",
        "dominant_interval_weight_fraction",
        "supported_event_span_hours",
    ]

    return (
        unit_base[
            keep_columns
        ].copy(),
        {
            "units": len(
                unit_base
            ),
            "points": len(
                weighted_points
            ),
            "max_weight_conservation_error_m": max_weight_error,
        },
    )


## 5. Incident-wise resumable weighted-evidence production

In [ ]:

RUN_LOG_PATH = (
    OUTPUT_ROOT
    / "03M_RUN_LOG.csv"
)

run_rows = []

outer = tqdm(
    incident_inventory.itertuples(
        index=False,
        name=None,
    ),
    total=len(
        incident_inventory
    ),
    desc="03M weighted evidence incidents",
)

inventory_columns = list(
    incident_inventory.columns
)

for values in outer:

    row = dict(
        zip(
            inventory_columns,
            values,
        )
    )

    irwin = str(
        row[
            "_irwin_norm"
        ]
    )

    output_path = (
        UNIT_METRIC_SHARD_ROOT
        / f"{irwin}.parquet"
    )

    start = time.perf_counter()

    try:

        if (
            RESUME
            and output_path.exists()
        ):

            metric_preview = pd.read_parquet(
                output_path,
                columns=[
                    "unit_uid",
                ],
            )

            run_rows.append(
                {
                    "_irwin_norm": irwin,
                    "IncidentName": row[
                        "IncidentName"
                    ],
                    "status": "SKIPPED_COMPLETE",
                    "units": len(
                        metric_preview
                    ),
                    "runtime_seconds": (
                        time.perf_counter()
                        - start
                    ),
                }
            )

            outer.set_postfix(
                {
                    "incident": str(
                        row[
                            "IncidentName"
                        ]
                    )[
                        :14
                    ],
                    "status": "resume",
                }
            )

            continue

        (
            unit_metrics,
            metadata,
        ) = audit_incident(
            unit_path=Path(
                row[
                    "unit_path"
                ]
            ),
            point_path=Path(
                row[
                    "point_path"
                ]
            ),
            evidence_path=Path(
                row[
                    "evidence_path"
                ]
            ),
            trajectory_path=Path(
                row[
                    "trajectory_path"
                ]
            ),
        )

        unit_metrics.to_parquet(
            output_path,
            index=False,
        )

        runtime = (
            time.perf_counter()
            - start
        )

        run_rows.append(
            {
                "_irwin_norm": irwin,
                "IncidentName": row[
                    "IncidentName"
                ],
                "status": "OK",
                "runtime_seconds": runtime,
                **metadata,
            }
        )

        outer.set_postfix(
            {
                "incident": str(
                    row[
                        "IncidentName"
                    ]
                )[
                    :13
                ],
                "units": metadata[
                    "units"
                ],
                "pts": metadata[
                    "points"
                ],
                "sec": f"{runtime:.2f}",
            }
        )

        del unit_metrics
        gc.collect()

    except Exception as exc:

        run_rows.append(
            {
                "_irwin_norm": irwin,
                "IncidentName": row[
                    "IncidentName"
                ],
                "status": "ERROR",
                "error": (
                    f"{type(exc).__name__}: {exc}"
                ),
                "runtime_seconds": (
                    time.perf_counter()
                    - start
                ),
            }
        )

        outer.set_postfix(
            {
                "incident": str(
                    row[
                        "IncidentName"
                    ]
                )[
                    :13
                ],
                "status": "ERROR",
            }
        )

    pd.DataFrame(
        run_rows
    ).to_csv(
        RUN_LOG_PATH,
        index=False,
    )


## 6. Combine weighted unit metrics

In [ ]:

metric_files = sorted(
    UNIT_METRIC_SHARD_ROOT.glob(
        "*.parquet"
    )
)

print(
    "Completed weighted-metric incident shards:",
    len(
        metric_files
    )
)

if (
    len(
        metric_files
    )
    != len(
        incident_inventory
    )
):

    print(
        "WARNING: weighted metric coverage is incomplete."
    )

metric_frames = []

for path in tqdm(
    metric_files,
    desc="Combine weighted unit metrics",
):

    metric_frames.append(
        pd.read_parquet(
            path
        )
    )

unit_metrics = (
    pd.concat(
        metric_frames,
        ignore_index=True,
    )
    if metric_frames
    else pd.DataFrame()
)

del metric_frames

if len(
    unit_metrics
):

    assert unit_metrics[
        "unit_uid"
    ].is_unique

print(
    "Weighted unit metrics:",
    f"{len(unit_metrics):,}"
)

COMBINED_UNIT_METRICS_PATH = (
    AUDIT_PROCESSED_ROOT
    / "weighted_unit_evidence_metrics.parquet"
)

unit_metrics.to_parquet(
    COMBINED_UNIT_METRICS_PATH,
    index=False,
)


## 7. Arc-length weighting validation

In [ ]:

weight_validation = pd.DataFrame(
    [
        {
            "units": len(
                unit_metrics
            ),
            "incidents": (
                unit_metrics[
                    "_irwin_norm"
                ].nunique()
                if len(
                    unit_metrics
                )
                else 0
            ),
            "max_weight_length_error_m": (
                unit_metrics[
                    "weight_length_abs_error_m"
                ].max()
                if len(
                    unit_metrics
                )
                else np.nan
            ),
            "p99_weight_length_error_m": (
                unit_metrics[
                    "weight_length_abs_error_m"
                ].quantile(
                    0.99
                )
                if len(
                    unit_metrics
                )
                else np.nan
            ),
            "mean_abs_difference_raw_vs_weighted_support_fraction": (
                (
                    unit_metrics[
                        "supported_fraction"
                    ]
                    - unit_metrics[
                        "weighted_support_fraction"
                    ]
                )
                .abs()
                .mean()
                if len(
                    unit_metrics
                )
                else np.nan
            ),
            "p95_abs_difference_raw_vs_weighted_support_fraction": (
                (
                    unit_metrics[
                        "supported_fraction"
                    ]
                    - unit_metrics[
                        "weighted_support_fraction"
                    ]
                )
                .abs()
                .quantile(
                    0.95
                )
                if len(
                    unit_metrics
                )
                else np.nan
            ),
        }
    ]
)

display(
    weight_validation
)

weight_validation.to_csv(
    OUTPUT_ROOT
    / "arc_length_weight_validation.csv",
    index=False,
)


## 8. Opportunity-bias audit by sample count

In [ ]:

SAMPLE_COUNT_BINS = [
    0,
    1,
    2,
    4,
    7,
    10,
    np.inf,
]

SAMPLE_COUNT_LABELS = [
    "1",
    "2",
    "3-4",
    "5-7",
    "8-10",
    ">10",
]

unit_metrics[
    "sample_count_bin"
] = pd.cut(
    unit_metrics[
        "sample_points"
    ],
    bins=SAMPLE_COUNT_BINS,
    labels=SAMPLE_COUNT_LABELS,
    right=True,
    include_lowest=True,
)

sample_count_audit = (
    unit_metrics.groupby(
        [
            "_engagement_norm",
            "sample_count_bin",
        ],
        observed=True,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        median_unit_length_m=(
            "actual_line_length_m",
            "median",
        ),
        primary_supported_pct=(
            "primary_supported",
            lambda s: (
                s.mean()
                * 100.0
            ),
        ),
        mean_weighted_support_pct=(
            "weighted_support_fraction",
            lambda s: (
                s.mean()
                * 100.0
            ),
        ),
        median_weighted_support_pct=(
            "weighted_support_fraction",
            lambda s: (
                s.median()
                * 100.0
            ),
        ),
        mean_weighted_conflict_pct=(
            "weighted_conflict_fraction",
            lambda s: (
                s.mean()
                * 100.0
            ),
        ),
        any_conflict_pct=(
            "weighted_conflict_fraction",
            lambda s: (
                (
                    s
                    > 0
                ).mean()
                * 100.0
            ),
        ),
    )
    .reset_index()
)

display(
    sample_count_audit
)

sample_count_audit.to_csv(
    OUTPUT_ROOT
    / "support_by_sample_count.csv",
    index=False,
)


## 9. Opportunity-bias audit by physical unit length

In [ ]:

LENGTH_BINS = [
    0.0,
    25.0,
    50.0,
    100.0,
    150.0,
    200.0,
    250.000001,
]

LENGTH_LABELS = [
    "<25 m",
    "25-50 m",
    "50-100 m",
    "100-150 m",
    "150-200 m",
    "200-250 m",
]

unit_metrics[
    "unit_length_bin"
] = pd.cut(
    unit_metrics[
        "actual_line_length_m"
    ],
    bins=LENGTH_BINS,
    labels=LENGTH_LABELS,
    right=False,
    include_lowest=True,
)

length_audit = (
    unit_metrics.groupby(
        [
            "_engagement_norm",
            "unit_length_bin",
        ],
        observed=True,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        median_sample_points=(
            "sample_points",
            "median",
        ),
        primary_supported_pct=(
            "primary_supported",
            lambda s: (
                s.mean()
                * 100.0
            ),
        ),
        mean_weighted_support_pct=(
            "weighted_support_fraction",
            lambda s: (
                s.mean()
                * 100.0
            ),
        ),
        median_weighted_support_pct=(
            "weighted_support_fraction",
            lambda s: (
                s.median()
                * 100.0
            ),
        ),
        mean_weighted_conflict_pct=(
            "weighted_conflict_fraction",
            lambda s: (
                s.mean()
                * 100.0
            ),
        ),
        any_conflict_pct=(
            "weighted_conflict_fraction",
            lambda s: (
                (
                    s
                    > 0
                ).mean()
                * 100.0
            ),
        ),
    )
    .reset_index()
)

display(
    length_audit
)

length_audit.to_csv(
    OUTPUT_ROOT
    / "support_by_unit_length.csv",
    index=False,
)



# 10. Rank-correlation opportunity diagnostics

These correlations are descriptive only.

They do **not** establish that sample count causes support because longer units and more complex fireline geometry can also differ physically.

The useful question is whether the primary rule exhibits a strong monotonic relationship with sampling opportunity that is reduced when evidence is expressed as arc-length fractions.


In [ ]:

correlation_rows = []

for (
    label,
    group,
) in unit_metrics.groupby(
    "_engagement_norm"
):

    if (
        len(
            group
        )
        < 3
    ):

        continue

    metrics = [
        (
            "sample_points_vs_primary_supported",
            group[
                "sample_points"
            ],
            group[
                "primary_supported"
            ].astype(
                int
            ),
        ),
        (
            "sample_points_vs_weighted_support_fraction",
            group[
                "sample_points"
            ],
            group[
                "weighted_support_fraction"
            ],
        ),
        (
            "unit_length_vs_primary_supported",
            group[
                "actual_line_length_m"
            ],
            group[
                "primary_supported"
            ].astype(
                int
            ),
        ),
        (
            "unit_length_vs_weighted_support_fraction",
            group[
                "actual_line_length_m"
            ],
            group[
                "weighted_support_fraction"
            ],
        ),
        (
            "sample_points_vs_weighted_conflict_fraction",
            group[
                "sample_points"
            ],
            group[
                "weighted_conflict_fraction"
            ],
        ),
    ]

    for (
        metric_name,
        x,
        y,
    ) in metrics:

        result = spearmanr(
            x,
            y,
            nan_policy="omit",
        )

        correlation_rows.append(
            {
                "_engagement_norm": label,
                "metric": metric_name,
                "spearman_rho": float(
                    result.statistic
                ),
                "p_value": float(
                    result.pvalue
                ),
                "units": len(
                    group
                ),
            }
        )

opportunity_correlations = pd.DataFrame(
    correlation_rows
)

display(
    opportunity_correlations
)

opportunity_correlations.to_csv(
    OUTPUT_ROOT
    / "opportunity_rank_correlations.csv",
    index=False,
)


## 11. Challenge-interval dispersion

In [ ]:

event_units = unit_metrics[
    unit_metrics[
        "supported_interval_count"
    ]
    > 0
].copy()

challenge_rows = []

for (
    label,
    group,
) in event_units.groupby(
    "_engagement_norm"
):

    challenge_rows.append(
        {
            "_engagement_norm": label,
            "units_with_supported_event": len(
                group
            ),
            "incidents": group[
                "_irwin_norm"
            ].nunique(),
            "one_supported_interval_pct": (
                (
                    group[
                        "supported_interval_count"
                    ]
                    == 1
                ).mean()
                * 100.0
            ),
            "median_supported_intervals": group[
                "supported_interval_count"
            ].median(),
            "p90_supported_intervals": group[
                "supported_interval_count"
            ].quantile(
                0.90
            ),
            "p95_supported_intervals": group[
                "supported_interval_count"
            ].quantile(
                0.95
            ),
            "dominant_interval_ge_50pct": (
                (
                    group[
                        "dominant_interval_weight_fraction"
                    ]
                    >= 0.50
                ).mean()
                * 100.0
            ),
            "dominant_interval_ge_75pct": (
                (
                    group[
                        "dominant_interval_weight_fraction"
                    ]
                    >= 0.75
                ).mean()
                * 100.0
            ),
            "dominant_interval_ge_90pct": (
                (
                    group[
                        "dominant_interval_weight_fraction"
                    ]
                    >= 0.90
                ).mean()
                * 100.0
            ),
            "median_dominant_interval_weight_pct": (
                group[
                    "dominant_interval_weight_fraction"
                ].median()
                * 100.0
            ),
            "median_supported_event_span_hours": group[
                "supported_event_span_hours"
            ].median(),
            "p90_supported_event_span_hours": group[
                "supported_event_span_hours"
            ].quantile(
                0.90
            ),
            "p95_supported_event_span_hours": group[
                "supported_event_span_hours"
            ].quantile(
                0.95
            ),
        }
    )

challenge_dispersion = pd.DataFrame(
    challenge_rows
)

display(
    challenge_dispersion
)

challenge_dispersion.to_csv(
    OUTPUT_ROOT
    / "challenge_interval_dispersion.csv",
    index=False,
)


## 12. Challenge dispersion by sample count

In [ ]:

challenge_by_samples = (
    event_units.groupby(
        [
            "_engagement_norm",
            "sample_count_bin",
        ],
        observed=True,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        one_interval_pct=(
            "supported_interval_count",
            lambda s: (
                (
                    s
                    == 1
                ).mean()
                * 100.0
            ),
        ),
        median_supported_intervals=(
            "supported_interval_count",
            "median",
        ),
        median_dominant_weight_pct=(
            "dominant_interval_weight_fraction",
            lambda s: (
                s.median()
                * 100.0
            ),
        ),
        median_event_span_hours=(
            "supported_event_span_hours",
            "median",
        ),
    )
    .reset_index()
)

display(
    challenge_by_samples
)

challenge_by_samples.to_csv(
    OUTPUT_ROOT
    / "challenge_dispersion_by_sample_count.csv",
    index=False,
)



# 13. Weighted evidence-rule sensitivity

For each support threshold \(s\) and Held conflict tolerance \(c\):

### Burned Over

\[
f_S\ge s.
\]

### Held

\[
f_S\ge s
\]

and

\[
f_C\le c.
\]

The table reports resulting cohort size, incident coverage, both-class incident count, and incident concentration.

The original 03L primary rule is included as the benchmark row.


In [ ]:

def concentration_from_mask(
    frame,
    mask,
):

    selected = (
        frame.loc[
            mask
        ]
    )

    if not len(
        selected
    ):

        return {
            "units": 0,
            "incidents": 0,
            "top1_pct": np.nan,
            "top10_pct": np.nan,
            "effective_incidents": np.nan,
        }

    counts = (
        selected.groupby(
            "_irwin_norm"
        )
        .size()
        .sort_values(
            ascending=False
        )
        .to_numpy(
            dtype=float
        )
    )

    total = counts.sum()

    return {
        "units": int(
            total
        ),
        "incidents": len(
            counts
        ),
        "top1_pct": (
            counts[
                :1
            ].sum()
            / total
            * 100.0
        ),
        "top10_pct": (
            counts[
                :min(
                    10,
                    len(
                        counts
                    )
                )
            ].sum()
            / total
            * 100.0
        ),
        "effective_incidents": (
            total**2
            / np.square(
                counts
            ).sum()
        ),
    }


def both_class_incidents(
    selected,
):

    if not len(
        selected
    ):

        return 0

    pivot = (
        selected.groupby(
            [
                "_irwin_norm",
                "_engagement_norm",
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    for column in [
        "Held",
        "Burned Over",
    ]:

        if column not in (
            pivot.columns
        ):

            pivot[
                column
            ] = 0

    return int(
        (
            (
                pivot[
                    "Held"
                ]
                > 0
            )
            & (
                pivot[
                    "Burned Over"
                ]
                > 0
            )
        ).sum()
    )


rule_rows = []

# ------------------------------------------------------------
# Benchmark: 03L primary any/veto rule
# ------------------------------------------------------------

primary_mask = (
    unit_metrics[
        "primary_supported"
    ]
)

primary_selected = unit_metrics[
    primary_mask
]

primary_all = concentration_from_mask(
    unit_metrics,
    primary_mask,
)

primary_held_mask = (
    primary_mask
    & (
        unit_metrics[
            "_engagement_norm"
        ]
        == "Held"
    )
)

primary_bo_mask = (
    primary_mask
    & (
        unit_metrics[
            "_engagement_norm"
        ]
        == "Burned Over"
    )
)

primary_held = concentration_from_mask(
    unit_metrics,
    primary_held_mask,
)

primary_bo = concentration_from_mask(
    unit_metrics,
    primary_bo_mask,
)

rule_rows.append(
    {
        "rule_name": "03L_primary_any_veto",
        "support_threshold": np.nan,
        "held_conflict_tolerance": np.nan,
        "total_supported_units": primary_all[
            "units"
        ],
        "total_supported_incidents": primary_all[
            "incidents"
        ],
        "held_supported_units": primary_held[
            "units"
        ],
        "held_supported_incidents": primary_held[
            "incidents"
        ],
        "burned_over_supported_units": primary_bo[
            "units"
        ],
        "burned_over_supported_incidents": primary_bo[
            "incidents"
        ],
        "incidents_with_both_classes": both_class_incidents(
            primary_selected
        ),
        "overall_top10_pct": primary_all[
            "top10_pct"
        ],
        "overall_effective_incidents": primary_all[
            "effective_incidents"
        ],
        "held_top10_pct": primary_held[
            "top10_pct"
        ],
        "held_effective_incidents": primary_held[
            "effective_incidents"
        ],
        "burned_over_top10_pct": primary_bo[
            "top10_pct"
        ],
        "burned_over_effective_incidents": primary_bo[
            "effective_incidents"
        ],
    }
)

# ------------------------------------------------------------
# Weighted coverage rules
# ------------------------------------------------------------

is_held = (
    unit_metrics[
        "_engagement_norm"
    ]
    == "Held"
)

is_bo = (
    unit_metrics[
        "_engagement_norm"
    ]
    == "Burned Over"
)

for support_threshold in (
    SUPPORT_THRESHOLDS
):

    bo_rule = (
        is_bo
        & (
            unit_metrics[
                "weighted_support_fraction"
            ]
            >= support_threshold
        )
    )

    for conflict_tolerance in (
        HELD_CONFLICT_TOLERANCES
    ):

        held_rule = (
            is_held
            & (
                unit_metrics[
                    "weighted_support_fraction"
                ]
                >= support_threshold
            )
            & (
                unit_metrics[
                    "weighted_conflict_fraction"
                ]
                <= (
                    conflict_tolerance
                    + 1e-12
                )
            )
        )

        combined = (
            held_rule
            | bo_rule
        )

        selected = unit_metrics[
            combined
        ]

        all_metrics = concentration_from_mask(
            unit_metrics,
            combined,
        )

        held_metrics = concentration_from_mask(
            unit_metrics,
            held_rule,
        )

        bo_metrics = concentration_from_mask(
            unit_metrics,
            bo_rule,
        )

        rule_rows.append(
            {
                "rule_name": (
                    f"weighted_s{support_threshold:.2f}"
                    f"_held_c{conflict_tolerance:.2f}"
                ),
                "support_threshold": support_threshold,
                "held_conflict_tolerance": conflict_tolerance,
                "total_supported_units": all_metrics[
                    "units"
                ],
                "total_supported_incidents": all_metrics[
                    "incidents"
                ],
                "held_supported_units": held_metrics[
                    "units"
                ],
                "held_supported_incidents": held_metrics[
                    "incidents"
                ],
                "burned_over_supported_units": bo_metrics[
                    "units"
                ],
                "burned_over_supported_incidents": bo_metrics[
                    "incidents"
                ],
                "incidents_with_both_classes": both_class_incidents(
                    selected
                ),
                "overall_top10_pct": all_metrics[
                    "top10_pct"
                ],
                "overall_effective_incidents": all_metrics[
                    "effective_incidents"
                ],
                "held_top10_pct": held_metrics[
                    "top10_pct"
                ],
                "held_effective_incidents": held_metrics[
                    "effective_incidents"
                ],
                "burned_over_top10_pct": bo_metrics[
                    "top10_pct"
                ],
                "burned_over_effective_incidents": bo_metrics[
                    "effective_incidents"
                ],
            }
        )

rule_sensitivity = pd.DataFrame(
    rule_rows
)

benchmark_units = int(
    rule_sensitivity.loc[
        rule_sensitivity[
            "rule_name"
        ]
        == "03L_primary_any_veto",
        "total_supported_units",
    ].iloc[
        0
    ]
)

benchmark_held = int(
    rule_sensitivity.loc[
        rule_sensitivity[
            "rule_name"
        ]
        == "03L_primary_any_veto",
        "held_supported_units",
    ].iloc[
        0
    ]
)

benchmark_bo = int(
    rule_sensitivity.loc[
        rule_sensitivity[
            "rule_name"
        ]
        == "03L_primary_any_veto",
        "burned_over_supported_units",
    ].iloc[
        0
    ]
)

rule_sensitivity[
    "total_change_vs_03L_pct"
] = (
    (
        rule_sensitivity[
            "total_supported_units"
        ]
        - benchmark_units
    )
    / benchmark_units
    * 100.0
)

rule_sensitivity[
    "held_change_vs_03L_pct"
] = (
    (
        rule_sensitivity[
            "held_supported_units"
        ]
        - benchmark_held
    )
    / benchmark_held
    * 100.0
)

rule_sensitivity[
    "burned_over_change_vs_03L_pct"
] = (
    (
        rule_sensitivity[
            "burned_over_supported_units"
        ]
        - benchmark_bo
    )
    / benchmark_bo
    * 100.0
)

display(
    rule_sensitivity
)

rule_sensitivity.to_csv(
    OUTPUT_ROOT
    / "weighted_rule_sensitivity.csv",
    index=False,
)


## 14. Supported-count sensitivity pivots

In [ ]:

weighted_only = rule_sensitivity[
    rule_sensitivity[
        "rule_name"
    ]
    != "03L_primary_any_veto"
].copy()

held_count_pivot = (
    weighted_only.pivot(
        index="support_threshold",
        columns="held_conflict_tolerance",
        values="held_supported_units",
    )
)

total_count_pivot = (
    weighted_only.pivot(
        index="support_threshold",
        columns="held_conflict_tolerance",
        values="total_supported_units",
    )
)

bo_count_by_support = (
    weighted_only[
        [
            "support_threshold",
            "burned_over_supported_units",
            "burned_over_supported_incidents",
            "burned_over_top10_pct",
            "burned_over_effective_incidents",
        ]
    ]
    .drop_duplicates(
        "support_threshold"
    )
    .sort_values(
        "support_threshold"
    )
)

display(
    held_count_pivot
)

display(
    total_count_pivot
)

display(
    bo_count_by_support
)

held_count_pivot.to_csv(
    OUTPUT_ROOT
    / "held_support_count_grid.csv"
)

total_count_pivot.to_csv(
    OUTPUT_ROOT
    / "total_support_count_grid.csv"
)

bo_count_by_support.to_csv(
    OUTPUT_ROOT
    / "burned_over_support_threshold_sensitivity.csv",
    index=False,
)



# 15. Stability diagnostics

The purpose of this section is not to pick a threshold mechanically.

Instead, it identifies whether neighboring support/conflict settings produce similar cohort sizes and incident coverage.

A broad stable region is preferable to a threshold that sits on a sharp cliff.


In [ ]:

stability_rows = []

for conflict_tolerance in (
    HELD_CONFLICT_TOLERANCES
):

    subset = (
        weighted_only[
            weighted_only[
                "held_conflict_tolerance"
            ]
            == conflict_tolerance
        ]
        .sort_values(
            "support_threshold"
        )
        .copy()
    )

    subset[
        "adjacent_total_change_pct"
    ] = (
        subset[
            "total_supported_units"
        ]
        .pct_change()
        .abs()
        * 100.0
    )

    subset[
        "adjacent_held_change_pct"
    ] = (
        subset[
            "held_supported_units"
        ]
        .pct_change()
        .abs()
        * 100.0
    )

    subset[
        "adjacent_bo_change_pct"
    ] = (
        subset[
            "burned_over_supported_units"
        ]
        .pct_change()
        .abs()
        * 100.0
    )

    stability_rows.append(
        subset
    )

threshold_stability = pd.concat(
    stability_rows,
    ignore_index=True,
)

display(
    threshold_stability[
        [
            "support_threshold",
            "held_conflict_tolerance",
            "total_supported_units",
            "held_supported_units",
            "burned_over_supported_units",
            "total_supported_incidents",
            "incidents_with_both_classes",
            "adjacent_total_change_pct",
            "adjacent_held_change_pct",
            "adjacent_bo_change_pct",
        ]
    ]
)

threshold_stability.to_csv(
    OUTPUT_ROOT
    / "weighted_threshold_stability.csv",
    index=False,
)


## 16. 03M decision table

In [ ]:

decision_table = pd.DataFrame(
    [
        {
            "question": "Arc-length weights conserve corrected unit length",
            "value": (
                weight_validation.iloc[
                    0
                ][
                    "max_weight_length_error_m"
                ]
            ),
            "interpretation": (
                "Maximum absolute line-length reconstruction error in meters."
            ),
        },
        {
            "question": "Mean raw-vs-weighted support-fraction difference",
            "value": (
                weight_validation.iloc[
                    0
                ][
                    "mean_abs_difference_raw_vs_weighted_support_fraction"
                ]
            ),
            "interpretation": (
                "Quantifies how much equal-point voting differs from line-length weighting."
            ),
        },
        {
            "question": "Held primary supported units",
            "value": benchmark_held,
            "interpretation": (
                "03L any-supported + zero-conflict benchmark."
            ),
        },
        {
            "question": "Burned Over primary supported units",
            "value": benchmark_bo,
            "interpretation": (
                "03L any-crossing benchmark."
            ),
        },
        {
            "question": "Held units with supported challenge event",
            "value": int(
                (
                    event_units[
                        "_engagement_norm"
                    ]
                    == "Held"
                ).sum()
            ),
            "interpretation": (
                "Units containing at least one supported sample-point event."
            ),
        },
        {
            "question": "Burned Over units with supported challenge event",
            "value": int(
                (
                    event_units[
                        "_engagement_norm"
                    ]
                    == "Burned Over"
                ).sum()
            ),
            "interpretation": (
                "Units containing at least one supported sample-point event."
            ),
        },
    ]
)

display(
    decision_table
)

decision_table.to_csv(
    OUTPUT_ROOT
    / "UNIT_EVIDENCE_DECISION_TABLE.csv",
    index=False,
)


## 17. Write report

In [ ]:

def df_to_md(
    dataframe,
    max_rows=200,
):

    if (
        dataframe is None
        or len(
            dataframe
        ) == 0
    ):

        return "_No rows._"

    try:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_markdown(
                index=False
            )
        )

    except Exception:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_string(
                index=False
            )
        )


report = f"""# Corrected Unit Evidence Rule Audit

Notebook build: `{NOTEBOOK_BUILD_ID}`

Generated: {datetime.now(timezone.utc).isoformat()}

## Purpose

Audit how the 03L multipoint evidence should be converted into robust corrected-unit targets.

03M performs no NIROPS rasterization.

## Weighted unit metrics

Units: **{len(unit_metrics):,}**

Incidents: **{unit_metrics['_irwin_norm'].nunique() if len(unit_metrics) else 0:,}**

Combined weighted metrics:

`{COMBINED_UNIT_METRICS_PATH}`

## Arc-length validation

{df_to_md(weight_validation)}

## Support by sample count

{df_to_md(sample_count_audit)}

## Support by physical unit length

{df_to_md(length_audit)}

## Opportunity rank correlations

{df_to_md(opportunity_correlations)}

## Challenge-interval dispersion

{df_to_md(challenge_dispersion)}

## Challenge dispersion by sample count

{df_to_md(challenge_by_samples)}

## Weighted rule sensitivity

{df_to_md(rule_sensitivity)}

## Threshold stability

{df_to_md(
    threshold_stability[
        [
            "support_threshold",
            "held_conflict_tolerance",
            "total_supported_units",
            "held_supported_units",
            "burned_over_supported_units",
            "total_supported_incidents",
            "incidents_with_both_classes",
            "adjacent_total_change_pct",
            "adjacent_held_change_pct",
            "adjacent_bo_change_pct",
        ]
    ]
)}

## Decision table

{df_to_md(decision_table)}

## Interpretation framework

### Opportunity effect

A strong monotonic change in the original 03L primary support rate with sample count, especially when arc-length-weighted support fractions are more stable, would indicate that the any/veto rule is partly influenced by sampling opportunity.

The diagnostic must not be interpreted causally because physical unit length and fire behavior can also covary.

### Arc-length coverage

Weighted support/conflict fractions should be preferred for threshold sensitivity because they represent fractions of actual local fireline length rather than fractions of sample points.

### Held

Candidate Held rules require:

\[
f_S\ge s
\]

and

\[
f_C\le c.
\]

The preferred region should preserve incident diversity and avoid a sharp threshold cliff.

### Burned Over

Candidate Burned Over rules require:

\[
f_S\ge s.
\]

A low but nonzero support-length threshold can require spatially meaningful crossing evidence while avoiding the raw `any point` opportunity effect.

### Challenge time

The dominant supported interval is selected by **arc-length-supported weight**, not by which sample point happened to produce the earliest event.

If most supported units have one interval or a dominant interval carrying most supported line weight, that interval is the natural candidate for later HRRR aggregation.

If interval dispersion is broad, weather should instead be summarized over the full supported interval span.

## Next step

Do not rerasterize NIROPS.

Review this report and choose:

1. a final length-weighted unit evidence rule;
2. the final challenge-interval representation.

Then freeze the corrected ML target cohort and proceed to 04A dynamic approach features.
"""

REPORT_PATH = (
    OUTPUT_ROOT
    / "UNIT_EVIDENCE_RULE_AUDIT_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print(
    "\nSaved:",
    REPORT_PATH
)


## 18. Output inventory

In [ ]:

print("=" * 84)
print("03M OUTPUTS")
print("=" * 84)

for path in sorted(
    OUTPUT_ROOT.rglob(
        "*"
    )
):

    if path.is_file():

        size = (
            f"{path.stat().st_size / (1024**2):,.2f} MB"
            if (
                path.stat().st_size
                >= 1024**2
            )
            else f"{path.stat().st_size / 1024:,.1f} KB"
        )

        print(
            f"{str(path.relative_to(OUTPUT_ROOT)):<72}"
            f"{size:>15}"
        )

print()
print(
    "Weighted per-incident metric shards:",
    len(
        list(
            UNIT_METRIC_SHARD_ROOT.glob(
                "*.parquet"
            )
        )
    ),
)

print(
    "Combined metrics:",
    COMBINED_UNIT_METRICS_PATH
)

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID
)
